# Mission EdStats : réduire le périmètre du dataset

Pour choisir les pays et les indicateurs cibles d'academy, il faut d'abord réduire le périmètre : `EdStatsCountry.csv` mélange des vrais pays et des agrégats (World, European Union...), et `EdStatsSeries.csv` compte des milliers d'indicateurs répartis en catégories. Ce notebook construit par le code la liste des vrais pays `country_codes` (section 2), liste les catégories d'indicateurs pour y choisir les nôtres (section 3), puis écrit les CSV de sortie de la réduction dans `data/output/` (section 4).


## 1. Mise en place

### 1.1 Dossiers

`INPUT_DIR` pointe vers `data/input/` (lecture seule), `OUTPUT_DIR` vers `data/output/`. Deux chemins candidats selon l'endroit de lancement ; `OUTPUT_DIR` est créé si absent.


In [1]:
import pandas as pd
import pycountry
import shutil
from pathlib import Path

candidates = [
    Path.cwd() / "data" / "input",         # from the project root
    Path.cwd() / ".." / "data" / "input",  # from notebooks/
]

INPUT_DIR = None
for candidate in candidates:
    if candidate.exists():
        INPUT_DIR = candidate.resolve()
        break
if INPUT_DIR is None:
    raise FileNotFoundError("data/input/ not found")

OUTPUT_DIR = INPUT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIR, OUTPUT_DIR


(PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/input'),
 PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output'))

### 1.2 Chargement brut

`load_csv_input()` lit un CSV de `data/input/` tel quel : `utf-8` pour les accents, colonnes « Unnamed » écartées (la virgule finale de chaque ligne crée une colonne vide, c'est un défaut du fichier).

Source : les CSV de `data/input/` sont le jeu de données EdStats (Banque mondiale), téléchargé avec `make download` (voir `README.md` du projet).


In [2]:
def load_csv_input(name):
    """Read the CSV `name` from INPUT_DIR as is (raw)."""
    return pd.read_csv(
        INPUT_DIR / name,
        encoding="utf-8",
        usecols=lambda column: "Unnamed" not in column,
    )


## 2. Réduction aux vrais pays


### 2.1 Le référentiel

`EdStatsCountry.csv` : une ligne par pays ou agrégat, avec son `Country Code` (ex. `FRA`). Aperçu des 3 premières lignes :


In [3]:
# raw load kept for display only
dfs = {"EdStatsCountry.csv": load_csv_input("EdStatsCountry.csv")}
dfs["EdStatsCountry.csv"].head(3)


,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


In [4]:
# separate load for computation
df = load_csv_input("EdStatsCountry.csv")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")


241 rows, 31 columns


### 2.2 Le critère : un code ISO 3166-1 alpha-3 officiel

`pycountry.countries` est la liste officielle ISO : 249 entrées, chacune avec un code alpha-3 (ex. `FRA`). On en extrait tous les codes dans une liste, puis `isin()` teste chaque ligne : un filtre booléen, aucun `iloc` sur des positions.

Source : liste officielle ISO 3166-1 via `pycountry` (249 codes officiels, alpha-2 et alpha-3).


In [5]:
# all official ISO 3166-1 alpha-3 codes, as a plain list
iso_codes = []
for country in pycountry.countries:
    iso_codes.append(country.alpha_3)

# boolean filter: True when the row's Country Code is in that list
is_country = df["Country Code"].isin(iso_codes)

country_codes = df.loc[is_country, "Country Code"].tolist()  # the real countries
print(f"{len(df)} rows in the reference, {len(country_codes)} real countries kept")
print(", ".join(sorted(country_codes)))


241 rows in the reference, 214 real countries kept
ABW, AFG, AGO, ALB, AND, ARE, ARG, ARM, ASM, ATG, AUS, AUT, AZE, BDI, BEL, BEN, BFA, BGD, BGR, BHR, BHS, BIH, BLR, BLZ, BMU, BOL, BRA, BRB, BRN, BTN, BWA, CAF, CAN, CHE, CHL, CHN, CIV, CMR, COD, COG, COL, COM, CPV, CRI, CUB, CUW, CYM, CYP, CZE, DEU, DJI, DMA, DNK, DOM, DZA, ECU, EGY, ERI, ESP, EST, ETH, FIN, FJI, FRA, FRO, FSM, GAB, GBR, GEO, GHA, GIB, GIN, GMB, GNB, GNQ, GRC, GRD, GRL, GTM, GUM, GUY, HKG, HND, HRV, HTI, HUN, IDN, IMN, IND, IRL, IRN, IRQ, ISL, ISR, ITA, JAM, JOR, JPN, KAZ, KEN, KGZ, KHM, KIR, KNA, KOR, KWT, LAO, LBN, LBR, LBY, LCA, LIE, LKA, LSO, LTU, LUX, LVA, MAC, MAF, MAR, MCO, MDA, MDG, MDV, MEX, MHL, MKD, MLI, MLT, MMR, MNE, MNG, MNP, MOZ, MRT, MUS, MWI, MYS, NAM, NCL, NER, NGA, NIC, NLD, NOR, NPL, NRU, NZL, OMN, PAK, PAN, PER, PHL, PLW, PNG, POL, PRI, PRK, PRT, PRY, PSE, PYF, QAT, ROU, RUS, RWA, SAU, SDN, SEN, SGP, SLB, SLE, SLV, SMR, SOM, SRB, SSD, STP, SUR, SVK, SVN, SWE, SWZ, SXM, SYC, SYR, TCA, TCD, TGO, THA,

### 2.3 Les codes écartés

Les lignes écartées, avec leur nom et leur `Region` si renseignée :


In [6]:
excluded = df.loc[~is_country, ["Country Code", "Short Name", "Region"]]
print(f"{len(excluded)} codes excluded, {excluded['Region'].notna().sum()} of them with a Region:")
excluded


27 codes excluded, 2 of them with a Region:


,Country Code,Short Name,Region
5,ARB,Arab World,NaN
35,CHI,Channel Islands,Europe & Central Asia
57,EAP,East Asia & Pacific (developing only),NaN
58,EAS,East Asia & Pacific (all income levels),NaN
59,ECA,Europe & Central Asia (developing only),NaN
60,ECS,Europe & Central Asia (all income levels),NaN
63,EMU,Euro area,NaN
68,EUU,European Union,NaN
89,HIC,High income,NaN
92,HPC,Heavily indebted poor countries (HIPC),NaN


### 2.4 Deux codes écartés qui ont pourtant une région

`CHI` (Channel Islands) et `XKX` (Kosovo) ont une `Region` mais pas de code ISO officiel. Les deux critères diffèrent sur 4 codes, vus sur les données :


In [7]:
has_region = df["Region"].notna()
print(f"{len(df)} rows: Region set {has_region.sum()}, ISO code {is_country.sum()}, both {(has_region & is_country).sum()}")
print(f"Region yes / ISO no: {df.loc[has_region & ~is_country, 'Country Code'].tolist()}")
print(f"Region no / ISO yes: {df.loc[~has_region & is_country, 'Country Code'].tolist()}")
print(f"neither: {(~has_region & ~is_country).sum()} rows")


241 rows: Region set 214, ISO code 214, both 212
Region yes / ISO no: ['CHI', 'XKX']
Region no / ISO yes: ['GIB', 'NRU']
neither: 25 rows


Cherchons ces noms dans la liste ISO (un dict nom -> code) :


In [8]:
# name -> alpha-3 code for the whole official ISO list
iso_by_name = {}
for c in pycountry.countries:
    iso_by_name[c.name] = c.alpha_3

for name in ["Channel Islands", "Kosovo", "Jersey", "Guernsey"]:
    print(f"{name}: {iso_by_name.get(name, 'not in the official ISO list')}")  # .get(key, default) returns the default if the key is missing


Channel Islands: not in the official ISO list
Kosovo: not in the official ISO list
Jersey: JEY
Guernsey: GGY


Sources externes (liens ci-dessous) : le Kosovo est un État au statut contesté, « Code ISO 3166-1 : aucun », le code `XK` n'étant « à usage personnel » (Wikipédia, « Kosovo ») ; les îles Anglo-Normandes forment deux États, Jersey et Guernesey (Wikipédia, « Îles Anglo-Normandes »), chacun avec son code ISO officiel (`JEY`, `GGY` ci-dessus). `CHI` est donc un agrégat, `XKX` n'a pas de code officiel : le critère ISO les écarte à raison.

- Wikipédia, « Kosovo » : https://fr.wikipedia.org/wiki/Kosovo
- Wikipédia, « Îles Anglo-Normandes » : https://fr.wikipedia.org/wiki/Îles_Anglo-Normandes


### 2.5 Nettoyage du référentiel

On applique la liste `country_codes` construite ci-dessus au référentiel `EdStatsCountry.csv` : le résultat, `cleaned`, est écrit dans `data/output/` en section 4. Les fichiers de `data/input/` ne sont que lus, jamais modifiés.

`isin()` teste chaque ligne : un filtre booléen sur la colonne `Country Code`, aucun `iloc` sur des positions. Bilan : lignes avant / après, puis codes écartés.

In [9]:
# boolean filter on the column, no iloc: keep only the real countries
cleaned = df[df["Country Code"].isin(country_codes)]

excluded_codes = sorted(set(df["Country Code"]) - set(country_codes))
print(f"{len(df)} rows in the input, {len(cleaned)} rows kept")
print(f"{len(excluded_codes)} codes excluded: {', '.join(excluded_codes)}")

241 rows in the input, 214 rows kept
27 codes excluded: ARB, CHI, EAP, EAS, ECA, ECS, EMU, EUU, HIC, HPC, LAC, LCN, LDC, LIC, LMC, LMY, MEA, MIC, MNA, NAC, OED, SAS, SSA, SSF, UMC, WLD, XKX


Aperçu des 3 premières lignes du référentiel nettoyé :

In [10]:
cleaned.head(3)

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


### 2.6 Propagation aux autres fichiers

Trois autres fichiers de `data/input/` portent une clé pays et reçoivent le même filtre `country_codes` (section 2) : `EdStatsData.csv` (colonne `Country Code`), `EdStatsFootNote.csv` et `EdStatsCountry-Series.csv` (colonne `CountryCode`). Comme en 2.5, `isin()` teste chaque ligne : filtre booléen, aucun `iloc`. Les résultats filtrés sont calculés ici, avec un bilan lignes avant / après par fichier ; l'écriture des CSV dans `data/output/` se fait en section 4.

`EdStatsData.csv` : une ligne par pays et par indicateur, la clé pays est dans `Country Code`.

In [11]:
# the country key of EdStatsData.csv is in the "Country Code" column
data = load_csv_input("EdStatsData.csv")
data_cleaned = data[data["Country Code"].isin(country_codes)]
print(f"EdStatsData.csv: {len(data)} rows in the input, {len(data_cleaned)} rows kept")

EdStatsData.csv: 886930 rows in the input, 784310 rows kept


`EdStatsFootNote.csv` : une ligne par pays, indicateur et année, la clé pays est dans `CountryCode`.

In [12]:
# the country key of EdStatsFootNote.csv is in the "CountryCode" column
footnote = load_csv_input("EdStatsFootNote.csv")
footnote_cleaned = footnote[footnote["CountryCode"].isin(country_codes)]
print(f"EdStatsFootNote.csv: {len(footnote)} rows in the input, {len(footnote_cleaned)} rows kept")

EdStatsFootNote.csv: 643638 rows in the input, 517413 rows kept


`EdStatsCountry-Series.csv` : une ligne par couple pays × indicateur, la clé pays est dans `CountryCode`.

In [13]:
# the country key of EdStatsCountry-Series.csv is in the "CountryCode" column
country_series = load_csv_input("EdStatsCountry-Series.csv")
country_series_cleaned = country_series[country_series["CountryCode"].isin(country_codes)]
print(f"EdStatsCountry-Series.csv: {len(country_series)} rows in the input, {len(country_series_cleaned)} rows kept")

EdStatsCountry-Series.csv: 613 rows in the input, 599 rows kept


`EdStatsSeries.csv` n'est pas filtré : c'est le dictionnaire des indicateurs, une ligne par indicateur (`Series Code`), il ne contient aucune clé pays. Le filtre `country_codes` ne s'y applique pas, le fichier est copié tel quel dans `data/output/` en section 4. Preuve en code, sur l'en-tête du fichier :

In [14]:
# read only the header line to list the columns of EdStatsSeries.csv
series_header = pd.read_csv(INPUT_DIR / "EdStatsSeries.csv", encoding="utf-8", nrows=0)
country_columns = [c for c in series_header.columns if "country" in c.lower()]
print(f"{len(series_header.columns)} columns, {len(country_columns)} with a country key: {country_columns}")

21 columns, 0 with a country key: []


### Bilan

214 vrais pays retenus dans `country_codes` sur les 241 lignes du référentiel, 27 écartés (agrégats + `CHI`/`XKX`, section 2.4). Le même filtre a été propagé aux trois autres fichiers qui portent une clé pays (section 2.6, bilans lignes avant / après en sortie de code). Les résultats filtrés (`cleaned`, `data_cleaned`, `footnote_cleaned`, `country_series_cleaned`) sont écrits dans `data/output/` en section 4. Prochaine étape : choisir les catégories d'indicateurs pertinentes (section 3 ci-dessous).

## 3. Les catégories d'indicateurs

Deuxième volet de la réduction : choisir les indicateurs. `EdStatsSeries.csv` est le référentiel des indicateurs : une ligne par indicateur, avec son code (`Series Code`), son nom (`Indicator Name`) et sa catégorie (`Topic`).


### 3.1 Le référentiel des indicateurs

Aperçu des 3 premières lignes, sur les colonnes utiles :


In [15]:
# raw load kept for display only
dfs["EdStatsSeries.csv"] = load_csv_input("EdStatsSeries.csv")
dfs["EdStatsSeries.csv"][["Series Code", "Topic", "Indicator Name"]].head(3)


,Series Code,Topic,Indicator Name
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...


In [16]:
# separate load for computation
series = load_csv_input("EdStatsSeries.csv")
print(f"{series.shape[0]} indicators, {series.shape[1]} columns")
print(f"{series['Topic'].nunique()} categories in the Topic column")


3665 indicators, 20 columns
37 categories in the Topic column


### 3.2 Liste des catégories, traduites en français

Chaque catégorie (`Topic`) avec son nombre d'indicateurs et sa traduction en français, de la plus fournie à la moins fournie. Les traductions suivent la terminologie française de la Banque mondiale (portail EdStats et sa version française : donnees.banquemondiale.org, blogs et rapports SABER en français). « Laber » est une coquille du fichier source pour « Labor » (emploi).

Source : portail EdStats, https://datatopics.worldbank.org/education/ (thèmes Learning Outcomes, Attainment, Equity, Expenditures, Policy, EMIS) et sa version française.


In [17]:
# one row per category, most indicators first
topics = series["Topic"].value_counts().rename_axis("Topic").reset_index(name="indicators")

# English topic -> French label, following the World Bank's French terminology
topic_fr = {
    "Learning Outcomes": "Acquis scolaires",
    "Attainment": "Niveau d'éducation",
    "Education Equality": "Égalité dans l'éducation",
    "Secondary": "Enseignement secondaire",
    "Primary": "Enseignement primaire",
    "Population": "Population",
    "Tertiary": "Enseignement supérieur",
    "Teachers": "Enseignants",
    "Expenditures": "Dépenses",
    "Engaging the Private Sector (SABER)": "Participation du secteur privé (SABER)",
    "Literacy": "Alphabétisation",
    "School Autonomy and Accountability (SABER)": "Autonomie et redevabilité des établissements scolaires (SABER)",
    "Teachers (SABER)": "Enseignants (SABER)",
    "Early Childhood Education": "Éducation de la petite enfance",
    "Education Management Information Systems (SABER)": "Systèmes de gestion de l'information dans l'éducation (SABER)",
    "Post-Secondary/Non-Tertiary": "Enseignement post-secondaire non supérieur",
    "School Finance (SABER)": "Financement des écoles (SABER)",
    "Tertiary Education (SABER)": "Enseignement supérieur (SABER)",
    "Student Assessment (SABER)": "Évaluation des élèves (SABER)",
    "Pre-Primary": "Éducation préprimaire",
    "Health: Population: Structure": "Santé : population : structure",
    "Early Child Development (SABER)": "Développement de la petite enfance (SABER)",
    "Workforce Development (SABER)": "Développement de la main-d'œuvre (SABER)",
    "Social Protection & Labor: Labor force structure": "Protection sociale et emploi : structure de la main-d'œuvre",
    "School Health and School Feeding (SABER)": "Santé scolaire et alimentation scolaire (SABER)",
    "Economic Policy & Debt: Purchasing power parity": "Politique économique et dette : parité de pouvoir d'achat",
    "Economic Policy & Debt: National accounts: US$ at current prices: Aggregate indicators": "Politique économique et dette : comptes nationaux : $US aux prix courants : indicateurs agrégés",
    "Laber": "Emploi",
    "Social Protection & Labor: Unemployment": "Protection sociale et emploi : chômage",
    "Infrastructure: Communications": "Infrastructure : communications",
    "Economic Policy & Debt: National accounts: US$ at constant 2010 prices: Aggregate indicators": "Politique économique et dette : comptes nationaux : $US aux prix constants de 2010 : indicateurs agrégés",
    "Background": "Contexte",
    "EMIS": "Systèmes de gestion de l'information dans l'éducation (EMIS)",
    "Economic Policy & Debt: National accounts: Atlas GNI & GNI per capita": "Politique économique et dette : comptes nationaux : RNB méthode Atlas et RNB par habitant",
    "Health: Risk factors": "Santé : facteurs de risque",
    "Health: Mortality": "Santé : mortalité",
    "Health: Population: Dynamics": "Santé : population : dynamique",
}

# map() looks up each English topic in the dict and adds the French label
topics["Catégorie (FR)"] = topics["Topic"].map(topic_fr)
print(f"{topics['Catégorie (FR)'].isna().sum()} catégories sans traduction sur {len(topics)}")


0 catégories sans traduction sur 37


In [18]:
# the table: French label first, then the original English topic and the count
topics[["Catégorie (FR)", "Topic", "indicators"]]


,Catégorie (FR),Topic,indicators
0,Acquis scolaires,Learning Outcomes,1046
1,Niveau d'éducation,Attainment,733
2,Égalité dans l'éducation,Education Equality,426
3,Enseignement secondaire,Secondary,256
4,Enseignement primaire,Primary,248
5,Population,Population,213
6,Enseignement supérieur,Tertiary,158
7,Enseignants,Teachers,137
8,Dépenses,Expenditures,93
9,Participation du secteur privé (SABER),Engaging the Private Sector (SABER),51


## 4. Écriture des CSV de sortie

Tous les CSV de sortie de la réduction du périmètre sont écrits ici, en fin de notebook, une fois le travail de réduction fait en amont (vrais pays, section 2 ; catégories d'indicateurs, section 3). `OUTPUT_DIR` (`data/output/`) est créé en section 1 si absent ; les fichiers de `data/input/` ne sont jamais modifiés.

### 4.1 Le référentiel pays nettoyé

`cleaned` (section 2.5) est écrit dans `data/output/EdStatsCountry.csv` :

In [19]:
# write the cleaned reference to data/output/ (data/input/ stays untouched)
output_file = OUTPUT_DIR / "EdStatsCountry.csv"
cleaned.to_csv(output_file, index=False)
print(f"wrote {len(cleaned)} rows to {output_file}")

wrote 214 rows to /Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output/EdStatsCountry.csv


Vérification en code : on recharge le CSV de sortie, aucun `Country Code` ne doit sortir de `country_codes`.

In [20]:
# reload the output CSV and check: no code outside the real-country list
reloaded = pd.read_csv(output_file, encoding="utf-8")
outside = reloaded.loc[~reloaded["Country Code"].isin(country_codes), "Country Code"].unique()
print(f"{len(reloaded)} rows in {output_file.name}, {len(outside)} codes outside country_codes: {list(outside)}")

214 rows in EdStatsCountry.csv, 0 codes outside country_codes: []


### 4.2 EdStatsData.csv

`data_cleaned` (section 2.6) est écrit dans `data/output/EdStatsData.csv` :

In [21]:
# write the filtered file to data/output/ (data/input/ stays untouched)
output_file = OUTPUT_DIR / "EdStatsData.csv"
data_cleaned.to_csv(output_file, index=False)
print(f"wrote {len(data_cleaned)} rows to {output_file}")

wrote 784310 rows to /Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output/EdStatsData.csv


Vérification en code : on recharge le CSV de sortie, aucun code pays ne doit sortir de `country_codes`.

In [22]:
# reload the output CSV and check: no code outside the real-country list
reloaded = pd.read_csv(output_file, encoding="utf-8")
outside = reloaded.loc[~reloaded["Country Code"].isin(country_codes), "Country Code"].unique()
print(f"{len(reloaded)} rows in {output_file.name}, {len(outside)} codes outside country_codes: {list(outside)}")

784310 rows in EdStatsData.csv, 0 codes outside country_codes: []


### 4.3 EdStatsFootNote.csv

`footnote_cleaned` (section 2.6) est écrit dans `data/output/EdStatsFootNote.csv` :

In [23]:
# write the filtered file to data/output/ (data/input/ stays untouched)
output_file = OUTPUT_DIR / "EdStatsFootNote.csv"
footnote_cleaned.to_csv(output_file, index=False)
print(f"wrote {len(footnote_cleaned)} rows to {output_file}")

wrote 517413 rows to /Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output/EdStatsFootNote.csv


Vérification en code : on recharge le CSV de sortie, aucun code pays ne doit sortir de `country_codes`.

In [24]:
# reload the output CSV and check: no code outside the real-country list
reloaded = pd.read_csv(output_file, encoding="utf-8")
outside = reloaded.loc[~reloaded["CountryCode"].isin(country_codes), "CountryCode"].unique()
print(f"{len(reloaded)} rows in {output_file.name}, {len(outside)} codes outside country_codes: {list(outside)}")

517413 rows in EdStatsFootNote.csv, 0 codes outside country_codes: []


### 4.4 EdStatsCountry-Series.csv

`country_series_cleaned` (section 2.6) est écrit dans `data/output/EdStatsCountry-Series.csv` :

In [25]:
# write the filtered file to data/output/ (data/input/ stays untouched)
output_file = OUTPUT_DIR / "EdStatsCountry-Series.csv"
country_series_cleaned.to_csv(output_file, index=False)
print(f"wrote {len(country_series_cleaned)} rows to {output_file}")

wrote 599 rows to /Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output/EdStatsCountry-Series.csv


Vérification en code : on recharge le CSV de sortie, aucun code pays ne doit sortir de `country_codes`.

In [26]:
# reload the output CSV and check: no code outside the real-country list
reloaded = pd.read_csv(output_file, encoding="utf-8")
outside = reloaded.loc[~reloaded["CountryCode"].isin(country_codes), "CountryCode"].unique()
print(f"{len(reloaded)} rows in {output_file.name}, {len(outside)} codes outside country_codes: {list(outside)}")

599 rows in EdStatsCountry-Series.csv, 0 codes outside country_codes: []


### 4.5 EdStatsSeries.csv

Pas de clé pays dans ce fichier (section 2.6) : il est copié tel quel, octet par octet, dans `data/output/` :

In [27]:
# no country key in this file: copy it as is, byte for byte
src = INPUT_DIR / "EdStatsSeries.csv"
dst = OUTPUT_DIR / "EdStatsSeries.csv"
shutil.copyfile(src, dst)
print(f"copied {src.name} to {dst.name}: {dst.stat().st_size} bytes, same size as the input: {src.stat().st_size == dst.stat().st_size}")

copied EdStatsSeries.csv to EdStatsSeries.csv: 3706781 bytes, same size as the input: True


## 5. Contrôles de cohérence

La section 4 vérifiait chaque CSV à son écriture. Cette section regroupe tous les contrôles en un seul endroit, sur les fichiers rechargés depuis `data/output/` : ils portent sur ce qui est écrit sur le disque, pas seulement sur les objets en mémoire. Objectif : aucun code pays hors de `country_codes` (anti-jointure à zéro ligne) et des clés uniques dans chaque sortie. Un tableau récapitule d'abord les lignes avant / après, par fichier.

### 5.1 Bilan : lignes avant / après

Pour chaque fichier : les lignes de `data/input/` (avant) et celles de `data/output/` (après). `EdStatsSeries.csv` n'étant pas filtré, avant et après doivent être égaux.

In [28]:
# reload every output CSV from data/output/: the controls run on the files on disk
outputs = {}
for name in ["EdStatsCountry.csv", "EdStatsData.csv", "EdStatsFootNote.csv", "EdStatsCountry-Series.csv", "EdStatsSeries.csv"]:
    outputs[name] = pd.read_csv(OUTPUT_DIR / name, encoding="utf-8", usecols=lambda column: "Unnamed" not in column)
    print(f"{name}: {len(outputs[name])} rows in data/output/")

EdStatsCountry.csv: 214 rows in data/output/


EdStatsData.csv: 784310 rows in data/output/
EdStatsFootNote.csv: 517413 rows in data/output/
EdStatsCountry-Series.csv: 599 rows in data/output/
EdStatsSeries.csv: 3665 rows in data/output/


In [29]:
# rows before (data/input/) and after (data/output/), one row per file
inputs = {
    "EdStatsCountry.csv": df,
    "EdStatsData.csv": data,
    "EdStatsFootNote.csv": footnote,
    "EdStatsCountry-Series.csv": country_series,
    "EdStatsSeries.csv": series,
}
summary = pd.DataFrame({
    "file": list(inputs.keys()),
    "rows before": [len(inputs[name]) for name in inputs],
    "rows after": [len(outputs[name]) for name in inputs],
})
print(f"{len(summary)} files: {summary['rows before'].sum()} rows before, {summary['rows after'].sum()} rows after")

5 files: 1535087 rows before, 1306201 rows after


In [30]:
# display only
summary

,file,rows before,rows after
0,EdStatsCountry.csv,241,214
1,EdStatsData.csv,886930,784310
2,EdStatsFootNote.csv,643638,517413
3,EdStatsCountry-Series.csv,613,599
4,EdStatsSeries.csv,3665,3665


### 5.2 Anti-jointures : zéro code hors des vrais pays

Une anti-jointure garde les lignes de la table de gauche qui n'ont aucun correspondant dans la table de droite. Ici, gauche = un CSV de sortie, droite = `country_codes` : le résultat doit être vide, aucun code d'agrégat (`WLD`, `EUU`...) ne doit avoir survécu au filtre de la section 2. `EdStatsSeries.csv` n'a pas de clé pays : on vérifie à la place qu'il est la copie conforme de l'entrée.

In [31]:
# anti-join per output file: keep only rows whose country code has NO match in country_codes
import filecmp

country_key = {
    "EdStatsCountry.csv": "Country Code",
    "EdStatsData.csv": "Country Code",
    "EdStatsFootNote.csv": "CountryCode",
    "EdStatsCountry-Series.csv": "CountryCode",
}
for name, key in country_key.items():
    ref = pd.DataFrame({key: country_codes})
    merged = outputs[name].merge(ref, on=key, how="left", indicator=True)
    outside = merged.loc[merged["_merge"] == "left_only"]  # left_only = no match on the right
    print(f"{name}: {len(outside)} row(s) with a code outside country_codes")
    assert len(outside) == 0  # the notebook stops here if an aggregate code survived

# EdStatsSeries.csv has no country key: it must be the input file copied as is
same = filecmp.cmp(INPUT_DIR / "EdStatsSeries.csv", OUTPUT_DIR / "EdStatsSeries.csv", shallow=False)
print(f"EdStatsSeries.csv: identical to the input: {same}")
assert same

EdStatsCountry.csv: 0 row(s) with a code outside country_codes
EdStatsData.csv: 0 row(s) with a code outside country_codes
EdStatsFootNote.csv: 0 row(s) with a code outside country_codes
EdStatsCountry-Series.csv: 0 row(s) with a code outside country_codes
EdStatsSeries.csv: identical to the input: True


### 5.3 Unicité des clés dans chaque sortie

Chaque fichier a une clé : les colonnes qui identifient chaque ligne (dictionnaire de `data_overview.ipynb`). Après le filtrage, chaque clé doit toujours apparaître sur une seule ligne. `duplicated(subset=...)` compte les clés en double.

In [32]:
# key of each output file
keys = {
    "EdStatsCountry.csv": ["Country Code"],
    "EdStatsData.csv": ["Country Code", "Indicator Code"],
    "EdStatsFootNote.csv": ["CountryCode", "SeriesCode", "Year"],
    "EdStatsCountry-Series.csv": ["CountryCode", "SeriesCode"],
}
for name, key in keys.items():
    duplicates = outputs[name].duplicated(subset=key).sum()
    print(f"{name}: {duplicates} duplicated key(s) on {key}")
    assert duplicates == 0  # the notebook stops here if a key appears twice

EdStatsCountry.csv: 0 duplicated key(s) on ['Country Code']
EdStatsData.csv: 0 duplicated key(s) on ['Country Code', 'Indicator Code']
EdStatsFootNote.csv: 0 duplicated key(s) on ['CountryCode', 'SeriesCode', 'Year']
EdStatsCountry-Series.csv: 0 duplicated key(s) on ['CountryCode', 'SeriesCode']


### 5.4 Les colonnes d'années d'`EdStatsData`

`EdStatsData.csv` porte 65 colonnes d'années : chaque année de 1970 à 2017, puis une tous les 5 ans de 2020 à 2100. Les dernières sont des projections, pas des mesures. Aucune n'est supprimée à ce stade : choisir les années utiles relève de la phase d'analyse, que le flux `data/input/` vers `data/output/` rend possible.

In [33]:
# the year columns of EdStatsData.csv: annual, then projections; all of them are kept
year_columns = outputs["EdStatsData.csv"].columns[4:]  # after the 4 identification columns
annual = [c for c in year_columns if int(c) <= 2017]
proj = [c for c in year_columns if int(c) >= 2020]
print(f"{len(year_columns)} year columns kept, none removed")
print(f"{len(annual)} annual ({annual[0]} to {annual[-1]}), {len(proj)} projections every 5 years ({proj[0]} to {proj[-1]})")

65 year columns kept, none removed
48 annual (1970 to 2017), 17 projections every 5 years (2020 to 2100)


### Bilan

Les 4 CSV filtrés de `data/output/` passent les contrôles : anti-jointures à 0 ligne (aucun code hors de `country_codes`) et clés uniques dans chaque sortie. `EdStatsSeries.csv` est la copie conforme de l'entrée. Les 65 colonnes d'années d'`EdStatsData` sont toutes conservées : le choix des années se fera dans la phase d'analyse.